In [1]:
import os
import sys
import pandas as pd

In [4]:
expo_name = 'NEW_qs'
file_path = f"../outputs/Qwen2.5-VL-7B-qinstruct_{expo_name}/Qwen2.5-VL-7B-qinstruct_{expo_name}_MMStar.xlsx"   # change to your file path
ori = pd.read_excel(file_path, engine="openpyxl")
ori.head(5)

,index,question,answer,category,l2_category,bench,A,B,C,D,prediction
0,0,Which option describe the object relationship ...,A,coarse perception,image scene and topic,MMBench,The suitcase is on the book.,The suitcase is beneath the cat.,The suitcase is beneath the bed.,The suitcase is beneath the book.,A. The suitcase is on the book.
1,1,What is the main feature in the background of ...,B,coarse perception,image scene and topic,SEEDBench_IMG,A park bench near the water.,A couple sitting on a bench.,A body of water and the Golden Gate Bridge.,A mountain in the distance.,The main feature in the background of the imag...
2,2,What seems to be the theme of the image?,D,coarse perception,image scene and topic,SEEDBench_IMG,Hanging Posters,Music,Home decoration,Playing guitar,B. Music
3,3,What is the most prominent feature in the image?,C,coarse perception,image scene and topic,SEEDBench_IMG,The skyline,The golf course,The trees,The person,The most prominent feature in the image is the...
4,4,What is the center of focus in the image?,A,coarse perception,image scene and topic,SEEDBench_IMG,A man writing in a book,A boy with his head in his hands surrounded by...,A cluttered desk with books and a pen,A stack of books covering a young man's face,B. A boy with his head in his hands surrounded...


In [5]:
expo_name = 'NEW_qs'
file_path = f"../outputs/Qwen2.5-VL-7B-qinstruct_{expo_name}/Qwen2.5-VL-7B-qinstruct_{expo_name}_MMStar_openai_result.xlsx"   # change to your file path
df_qs = pd.read_excel(file_path, engine="openpyxl")

In [6]:
expo_name = 'NEW_qa'
file_path = f"../outputs/Qwen2.5-VL-7B-qinstruct_{expo_name}/Qwen2.5-VL-7B-qinstruct_{expo_name}_MMStar_openai_result.xlsx"   # change to your file path
df_qa = pd.read_excel(file_path, engine="openpyxl")

In [7]:
compare_cols = ['question', 'prediction', 'hit', 'log']

df_merged = pd.merge(
    df_qa, 
    df_qs[compare_cols], 
    on='question', 
    suffixes=('_qa', '_qs')
)

worse_samples = df_merged[(df_merged['hit_qa'] == 1) & (df_merged['hit_qs'] == 0)]
better_samples = df_merged[(df_merged['hit_qa'] == 0) & (df_merged['hit_qs'] == 1)]

print(f"Number of samples that became worse: {len(worse_samples)}")
print(f"Number of samples that became better: {len(better_samples)}")
worse_samples.head(5)

Number of samples that became worse: 1206
Number of samples that became better: 504


,index,question,answer,category,l2_category,bench,A,B,C,D,prediction_qa,hit_qa,log_qa,prediction_qs,hit_qs,log_qs
26,13,Which mood does this image convey?,B,coarse perception,image emotion,MMBench,Sad,Cozy,Happy,Angry,B. Cozy,1,Match Log: B. Cozy.,The image conveys a mood of coziness and tranq...,0,"Match Log: Failed in Prefetch, no GPT-based an..."
27,13,Which mood does this image convey?,B,coarse perception,image emotion,MMBench,Sad,Cozy,Happy,Angry,B. Cozy,1,Match Log: B. Cozy.,"The image conveys a mood of intense emotion, p...",0,"Match Log: Failed in Prefetch, no GPT-based an..."
32,13,Which mood does this image convey?,B,coarse perception,image emotion,MMBench,Sad,Cozy,Happy,Angry,B. Cozy,1,Match Log: B. Cozy.,"The image conveys a mood of sadness, as the ch...",0,"Match Log: Failed in Prefetch, no GPT-based an..."
33,13,Which mood does this image convey?,B,coarse perception,image emotion,MMBench,Sad,Cozy,Happy,Angry,B. Cozy,1,Match Log: B. Cozy.,The image conveys a mood that could be interpr...,0,"Match Log: Failed in Prefetch, no GPT-based an..."
56,24,Which action is performed in this image?,C,coarse perception,image scene and topic,MMBench,marching,playing cymbals,long jump,cheerleading,C.,1,Match Log: C..,The image does not provide enough clarity to a...,0,"Match Log: Failed in Prefetch, no GPT-based an..."


In [8]:
# Save worsened samples to CSV
worse_samples.to_csv('worsened_samples.csv', index=False)

# Save both better and worse samples to an Excel file with multiple sheets
with pd.ExcelWriter('model_analysis.xlsx') as writer:
    worse_samples.to_excel(writer, sheet_name='Worsened', index=False)
    better_samples.to_excel(writer, sheet_name='Improved', index=False)